# Modelo Causal v7 — IPTW Corrigido

Este notebook corrige os dois problemas identificados no v6:

## Problemas do v6 e correções aplicadas

### Problema 1 — `frete_alto`: propensity score instável
| Item | v6 | v7 |
|---|---|---|
| Modelo PS | Regressão Logística | **Gradient Boosting** |
| Confundidores | `total_price` incluído | **`total_price` removido** (colinear: frete = freight/price) |
| Pesos | max=55.4 (instável) | trimming aplicado (percentil 99) |

### Problema 2 — `pedido_grande`: confundidor = base do tratamento
| Item | v6 | v7 |
|---|---|---|
| `n_items` nos confundidores | (errado) | **Removido** (`pedido_grande = n_items > 1`) |
| SMD de `n_items` | 1.93 antes e depois | Não se aplica — variável removida |

## Análises neste notebook
Foco nas 4 combinações que tiveram baixa confiabilidade no v6:
- `frete_alto → entrega_atrasada`
- `frete_alto → review_positivo`
- `pedido_grande → entrega_atrasada`
- `pedido_grande → review_positivo`

## 1. Imports

In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier   # muito mais rapido que GBM
from sklearn.metrics import roc_auc_score

from app.config.settings import INTERIM_DATA_DIR

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carregamento e preparação dos dados

In [11]:
df = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, "interim_dataset.parquet"))

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["approval_time_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df["dispatch_time_days"] = (
    df["order_delivered_carrier_date"] - df["order_approved_at"]
).dt.days

df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

df["perc_freight"] = df["total_freight"] / df["total_price"]

# Tratamentos
df["frete_alto"]    = (df["perc_freight"] > 0.2).astype(int)
df["pedido_grande"] = (df["n_items"] > 1).astype(int)

# Outcomes
df["entrega_atrasada"] = (df["delay_days"] > 0).astype(int)
df["review_positivo"]  = (df["review_score"] >= 4).astype(int)

print(f"Shape: {df.shape}")

Shape: (97712, 44)


## 3. Confundidores corrigidos

### Para `frete_alto`
- **Removido**: `total_price` — `frete_alto = total_freight / total_price > 0.2`, logo `total_price` determina diretamente o tratamento (colinearidade estrutural)
- **Mantido**: `avg_weight`, `customer_state` e demais — são causas independentes do frete

### Para `pedido_grande`
- **Removido**: `n_items` — `pedido_grande = (n_items > 1)` é uma transformação direta de `n_items`. Incluí-lo como confundidor é incluir o próprio tratamento
- **Mantido**: `total_price` — valor total pode ser confundidor independente do número de itens

In [12]:
# Confundidores originais (v6)
CONF_BASE = [
    "total_price", "n_items", "avg_weight",
    "customer_state", "purchase_month",
    "purchase_weekday", "purchase_hour", "n_items_missing_info"
]

# v7: frete_alto — remove total_price (colinear) e n_items
CONF_FRETE = [
    "avg_weight", "avg_price",
    "customer_state", "purchase_month",
    "purchase_weekday", "purchase_hour", "n_items_missing_info"
]

# v7: pedido_grande — remove n_items (base do tratamento)
CONF_PEDIDO = [
    "total_price", "avg_weight",
    "customer_state", "purchase_month",
    "purchase_weekday", "purchase_hour", "n_items_missing_info"
]

print("Confundidores frete_alto:",  CONF_FRETE)
print("Confundidores pedido_grande:", CONF_PEDIDO)

Confundidores frete_alto: ['avg_weight', 'avg_price', 'customer_state', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'n_items_missing_info']
Confundidores pedido_grande: ['total_price', 'avg_weight', 'customer_state', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'n_items_missing_info']


## 4. Funções IPTW v7

Novidades em relação ao v6:
- `model='gbm'` usa Gradient Boosting como estimador do propensity score
- `trim_percentile` corta pesos extremos no percentil especificado (padrão 99)
- Log do AUC-ROC do propensity score para avaliar qualidade do modelo

In [13]:
def preprocess(df, treatment, outcome, confundidores):
    cols = confundidores + [treatment, outcome]
    df_out = df[cols].dropna().copy()
    le = LabelEncoder()
    df_out['customer_state'] = le.fit_transform(df_out['customer_state'])
    continuous_cols = [c for c in confundidores if c != 'customer_state']
    scaler = StandardScaler()
    df_out[continuous_cols] = scaler.fit_transform(df_out[continuous_cols])
    return df_out


def compute_iptw_weights(df_model, treatment, confundidores,
                         stabilized=True, model='logistic', trim_percentile=99):
    """
    model: 'logistic' ou 'hgbm' (HistGradientBoosting — rapido e preciso)
    trim_percentile: corta pesos extremos no percentil especificado
    """
    X = df_model[confundidores].values
    T = df_model[treatment].values

    if model == 'hgbm':
        # HistGradientBoostingClassifier: ~10x mais rapido que GradientBoostingClassifier
        clf = HistGradientBoostingClassifier(
            max_iter=100, max_depth=4,
            learning_rate=0.05, random_state=42
        )
    else:
        clf = LogisticRegression(max_iter=1000, random_state=42)

    clf.fit(X, T)
    ps = clf.predict_proba(X)[:, 1]
    ps = np.clip(ps, 0.01, 0.99)

    auc = roc_auc_score(T, ps)

    p_treated = T.mean()
    weights = np.where(
        T == 1,
        (p_treated / ps) if stabilized else (1 / ps),
        ((1 - p_treated) / (1 - ps)) if stabilized else (1 / (1 - ps))
    )

    if trim_percentile is not None:
        w_max = np.percentile(weights, trim_percentile)
        weights = np.clip(weights, None, w_max)

    return ps, weights, auc


def compute_smd(df_model, treatment, confundidores, weights=None):
    smds = {}
    T = df_model[treatment].values
    for col in confundidores:
        x = df_model[col].values
        if weights is not None:
            mean1 = np.average(x[T == 1], weights=weights[T == 1])
            mean0 = np.average(x[T == 0], weights=weights[T == 0])
            var1  = np.average((x[T == 1] - mean1)**2, weights=weights[T == 1])
            var0  = np.average((x[T == 0] - mean0)**2, weights=weights[T == 0])
        else:
            mean1, var1 = x[T == 1].mean(), x[T == 1].var()
            mean0, var0 = x[T == 0].mean(), x[T == 0].var()
        pooled_std = np.sqrt((var1 + var0) / 2)
        smds[col] = abs(mean1 - mean0) / pooled_std if pooled_std > 0 else 0
    return pd.Series(smds)


def ate_iptw(df_model, treatment, outcome, weights):
    T = df_model[treatment].values
    Y = df_model[outcome].values
    return (
        np.average(Y[T == 1], weights=weights[T == 1]) -
        np.average(Y[T == 0], weights=weights[T == 0])
    )


def bootstrap_ci(df_model, treatment, outcome, confundidores,
                 n_bootstrap=200, alpha=0.05, model='logistic', trim_percentile=99):
    """
    Bootstrap usa sempre logistic — rapido e suficiente para variabilidade amostral.
    hgbm so e usado no ATE pontual principal.
    n_bootstrap=200 e suficiente para IC 95% estavel.
    """
    ates = []
    n = len(df_model)
    for _ in range(n_bootstrap):
        sample = df_model.sample(n=n, replace=True)
        _, w, _ = compute_iptw_weights(sample, treatment, confundidores,
                                       model='logistic',
                                       trim_percentile=trim_percentile)
        ates.append(ate_iptw(sample, treatment, outcome, w))
    lower = np.percentile(ates, 100 * alpha / 2)
    upper = np.percentile(ates, 100 * (1 - alpha / 2))
    return lower, upper, np.array(ates)

In [14]:
def run_iptw_analysis(df, treatment, outcome, confundidores,
                      model='logistic', trim_percentile=99,
                      ate_v6=None, assoc_bruta=None, n_bootstrap=200):
    print(f"{'='*60}")
    print(f"IPTW v7: {treatment} -> {outcome}  [modelo PS: {model}]")
    print(f"{'='*60}")

    df_model = preprocess(df, treatment, outcome, confundidores)
    print(f"N apos dropna: {len(df_model):,}")

    ps, weights, auc = compute_iptw_weights(
        df_model, treatment, confundidores,
        model=model, trim_percentile=trim_percentile
    )
    print(f"\nAUC-ROC propensity score: {auc:.4f}")
    print(f"PS     - media: {ps.mean():.3f} | min: {ps.min():.3f} | max: {ps.max():.3f}")
    print(f"Pesos  - media: {weights.mean():.3f} | min: {weights.min():.3f} | max: {weights.max():.3f}")

    smd_antes  = compute_smd(df_model, treatment, confundidores)
    smd_depois = compute_smd(df_model, treatment, confundidores, weights=weights)
    df_smd = pd.DataFrame({
        'SMD antes':  smd_antes.round(4),
        'SMD depois': smd_depois.round(4),
        'Balanceado?': smd_depois.apply(lambda x: '✅' if x < 0.1 else '❌')
    })
    print("\n--- Balanco das covariaveis (SMD < 0.1 = balanceado) ---")
    print(df_smd.to_string())

    ate = ate_iptw(df_model, treatment, outcome, weights)
    print(f"\nATE (IPTW v7):    {ate:+.6f}")
    if ate_v6 is not None:
        print(f"ATE (IPTW v6):    {ate_v6:+.6f}  (diff: {ate - ate_v6:+.6f})")
    if assoc_bruta is not None:
        print(f"Assoc. bruta EDA: {assoc_bruta:+.6f}")

    print(f"Calculando IC via bootstrap ({n_bootstrap} amostras)...")
    lower, upper, boot_ates = bootstrap_ci(
        df_model, treatment, outcome, confundidores,
        n_bootstrap=n_bootstrap, model=model, trim_percentile=trim_percentile
    )
    print(f"IC 95%: [{lower:.6f}, {upper:.6f}]")
    significativo = (lower > 0 or upper < 0)
    print(f"Resultado: {'✅ Significativo' if significativo else '❌ Nao significativo (IC contem zero)'}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(boot_ates, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0].axvline(ate,   color='red',    linewidth=2,   label=f'ATE = {ate:.4f}')
    axes[0].axvline(lower, color='orange', linewidth=1.5, linestyle='--', label=f'IC inf = {lower:.4f}')
    axes[0].axvline(upper, color='orange', linewidth=1.5, linestyle='--', label=f'IC sup = {upper:.4f}')
    axes[0].axvline(0,     color='black',  linewidth=1,   linestyle=':')
    if ate_v6:
        axes[0].axvline(ate_v6, color='green', linewidth=1.5,
                        linestyle='-.', label=f'ATE v6 = {ate_v6:.4f}')
    axes[0].set_title(f'Bootstrap ATE v7\n{treatment} -> {outcome} [{model}]')
    axes[0].set_xlabel('ATE estimado')
    axes[0].set_ylabel('Frequencia')
    axes[0].legend(fontsize=8)
    axes[0].grid(axis='y', linestyle='--', alpha=0.4)

    x = np.arange(len(confundidores))
    axes[1].barh(x - 0.2, smd_antes.values,  0.4, label='Antes',  color='coral',     alpha=0.8)
    axes[1].barh(x + 0.2, smd_depois.values, 0.4, label='Depois', color='steelblue', alpha=0.8)
    axes[1].axvline(0.1, color='red', linestyle='--', linewidth=1, label='Limite 0.1')
    axes[1].set_yticks(x)
    axes[1].set_yticklabels(confundidores, fontsize=8)
    axes[1].set_xlabel('SMD')
    axes[1].set_title('Balanco das covariaveis')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    fname = f'../../reports/figures/iptw_v7_{treatment}_{outcome}.png'
    plt.savefig(fname, dpi=150)
    plt.show()

    return {
        'treatment': treatment, 'outcome': outcome,
        'model_ps': model, 'N': len(df_model), 'auc': auc,
        'assoc_bruta': assoc_bruta, 'ate_v6': ate_v6,
        'ate': ate, 'ic_lower': lower, 'ic_upper': upper,
        'significativo': significativo
    }

## 5. Análises corrigidas — `frete_alto`

### 5.1 — Frete alto → Entrega atrasada

In [8]:
r1 = run_iptw_analysis(
    df, 'frete_alto', 'entrega_atrasada',
    confundidores=CONF_FRETE,
    model='hgbm',
    trim_percentile=99,
    n_bootstrap=200,
    ate_v6=0.013537,
    assoc_bruta=+0.0033
)

IPTW v7: frete_alto → entrega_atrasada  [modelo PS: gbm]
N após dropna: 96,929

AUC-ROC propensity score: 0.9626
PS     — média: 0.553 | min: 0.010 | max: 0.990
Pesos  — média: 0.743 | min: 0.450 | max: 5.401

--- Balanço das covariáveis (SMD < 0.1 = balanceado) ---
                      SMD antes  SMD depois Balanceado?
avg_weight               0.1634      0.1112           ❌
avg_price                0.8064      0.5314           ❌
customer_state           0.2546      0.1319           ❌
purchase_month           0.0125      0.0029           ✅
purchase_weekday         0.0118      0.0084           ✅
purchase_hour            0.0187      0.0224           ✅
n_items_missing_info     0.0114      0.0143           ✅

ATE (IPTW v7):    +0.004555
ATE (IPTW v6):    +0.013537  (diff: -0.008982)
Assoc. bruta EDA: +0.003300
Calculando IC via bootstrap (500 amostras)...


KeyboardInterrupt: 

### 5.2 — Frete alto → Review positivo

In [ ]:
r2 = run_iptw_analysis(
    df, 'frete_alto', 'review_positivo',
    confundidores=CONF_FRETE,
    model='hgbm',
    trim_percentile=99,
    n_bootstrap=200,
    ate_v6=-0.049713,
    assoc_bruta=-0.0051
)

## 6. Análises corrigidas — `pedido_grande`

### 6.1 — Pedido grande → Entrega atrasada

In [ ]:
r3 = run_iptw_analysis(
    df, 'pedido_grande', 'entrega_atrasada',
    confundidores=CONF_PEDIDO,
    model='logistic',
    trim_percentile=99,
    n_bootstrap=200,
    ate_v6=-0.016338,
    assoc_bruta=-0.0157
)

### 6.2 — Pedido grande → Review positivo

In [ ]:
r4 = run_iptw_analysis(
    df, 'pedido_grande', 'review_positivo',
    confundidores=CONF_PEDIDO,
    model='logistic',
    trim_percentile=99,
    n_bootstrap=200,
    ate_v6=-0.169202,
    assoc_bruta=-0.1633
)

## 7. Comparação: Assoc. bruta vs. v6 vs. v7

In [ ]:
resultados = [r1, r2, r3, r4]
df_comp = pd.DataFrame(resultados)

print("Comparação: Assoc. bruta (EDA) → ATE v6 → ATE v7")
print("=" * 85)
cols_show = ["treatment", "outcome", "model_ps", "auc",
             "assoc_bruta", "ate_v6", "ate", "ic_lower", "ic_upper", "significativo"]
print(df_comp[cols_show].rename(columns={
    "treatment":   "Tratamento",
    "outcome":     "Outcome",
    "model_ps":    "Modelo PS",
    "auc":         "AUC",
    "assoc_bruta": "Assoc. bruta",
    "ate_v6":      "ATE v6",
    "ate":         "ATE v7",
    "ic_lower":    "IC inf",
    "ic_upper":    "IC sup",
    "significativo": "Sig?"
}).to_string(index=False))

In [ ]:
# Gráfico comparativo: Assoc. bruta vs ATE v6 vs ATE v7
labels = [f"{r['treatment']}\n→ {r['outcome']}" for r in resultados]
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, df_comp["assoc_bruta"], width,
       label="Assoc. bruta (EDA)", color="coral",       alpha=0.85)
ax.bar(x,         df_comp["ate_v6"],     width,
       label="ATE IPTW v6",          color="#aec6e8",   alpha=0.85)
ax.bar(x + width, df_comp["ate"],        width,
       label="ATE IPTW v7 (corrigido)", color="steelblue", alpha=0.85)

# IC v7
ic_lower_err = df_comp["ate"] - df_comp["ic_lower"]
ic_upper_err = df_comp["ic_upper"] - df_comp["ate"]
ax.errorbar(x + width, df_comp["ate"],
            yerr=[ic_lower_err, ic_upper_err],
            fmt='none', color='navy', capsize=4, linewidth=1.5)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Efeito estimado")
ax.set_title("Assoc. bruta vs. ATE IPTW v6 vs. v7 (corrigido)\n"
             "(barras de erro = IC 95% bootstrap do v7)", fontsize=12)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("../../reports/figures/iptw_v7_comparacao_final.png", dpi=150)
plt.show()